# 01 · Descarga y modelos

Baja los registros de *Sepia officinalis* de iNaturalist para una **corrida** (un rectángulo y una carpeta en Drive) y los pasa por el **triaje** y los **4 prompts** de rasgos por foto. Este notebook **no muestra** los resultados de los modelos hasta la última sección, para no romper una anotación a ciegas posterior.

- Cada corrida tiene su config en `configs/` (por ejemplo `a2_test.yaml`) y sus datos en `MyDrive/sepia-inaturalist/data/<corrida>/`.
- Los registros que ya están en otra corrida **no se vuelven a bajar** (`skip_ids_from` en el config); quedan listados en `logs/skipped_existing.jsonl`.
- Todo es reanudable: si una celda se corta (por ejemplo, por el límite diario de Gemini), se vuelve a correr y sigue donde quedó.
- **Una corrida por sector de Martin** (11 nov): `configs/sector_a.yaml` … `sector_e.yaml`. El rectángulo y las zonas excluidas de cada sector están en `configs/sectors.yaml` (esquina noreste exacta, de la URL de Martin; esquina suroeste estimada de sus capturas, **pendiente** pedirle las URLs completas).
- Cada registro recibe su sector (`site`). Los de **otro** sector no entran a la corrida (`logs/other_sector.jsonl`; los baja la corrida de su sector). Los que caen **fuera de todo sector** o en una **zona excluida** (en amarillo en el PPT) quedan en la tabla con `excluded_reason`, sin fotos ni modelos, y salen en la hoja `excluded` del Excel.
- Para cambiar de sector, solo se cambia `CONFIG` abajo.

| Notebook | Para qué |
|---|---|
| `01_descarga_y_modelos` | bajar registros de iNaturalist y correr el triaje y los 4 prompts |
| `02_anotacion` | anotar: asistida (elegir el mejor valor de los modelos), revisión con letra y recuadro, o a ciegas |
| `03_revision_y_metricas` | revisión interna: ranking de prompts, moda, métricas, persona vs modelos, duplicados |
| `04_revision_con_martin` | lo que se le muestra a Martin: qué hicimos, resultados, fotos, preguntas y el Excel |
| `05_entrega_excel` | generar y revisar el Excel de entrega |

La versión oficial de cada notebook está en GitHub (`notebooks/`, sin resultados). Para trabajar: abrirlo desde GitHub en Colab, *Archivo → Guardar una copia en Drive* y moverla a `MyDrive/sepia-inaturalist/notebooks/` (con la fecha en el nombre).

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # nunca en el codigo

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'
BONEBOX = 'bonebox_v1'   # recuadro del hueso (solo fotos con 1 hueso segun v14)

In [ ]:
CONFIG = 'configs/sector_b.yaml'      # <- corrida a descargar y procesar
# Sectores: sector_a (solo lo que falta de A: A1 y A2 ya estan en pilot_a1 y a2_test),
#           sector_b, sector_c, sector_d, sector_e
cfg = sepia.load_config(CONFIG)
print(cfg.base_dir, '| sector:', cfg.sector, '| rectangulo:', cfg.bbox, '| salta registros de:', cfg.skip_ids_from)


## 0. Solo una vez: sectores en las corridas viejas (`pilot_a1`, `a2_test`)

Rehace las tablas desde la copia cruda guardada (sin internet, no baja nada) para que sus registros tengan `site`. **Ojo:** los registros que queden fuera de sector o en una zona excluida (por ejemplo, Thurso y la costa norte de Escocia, que quedan fuera del borde norte de A, pendiente con Martin) salen de la cola de anotación; sus anotaciones no se borran y aparecen en la hoja `excluded` del Excel. Los registros de A2 al sur de 52,67° N (por ejemplo Great Yarmouth) quedan con `site = B` pero se quedan en `a2_test`.

In [ ]:
for old in ['configs/pilot_a1.yaml', 'configs/a2_test.yaml']:
    print(f"\n===== {old} =====")
    !python scripts/download.py --config {old} --steps tables


## 1. ¿Cuántos registros?

Total de la consulta a iNaturalist. Incluye los que ya están en otras corridas: esos se descuentan en el paso 2.
Es el total del **rectángulo**: incluye registros de otros sectores y de zonas excluidas, que se descuentan en el paso 2.

In [ ]:
!python scripts/download.py --config {CONFIG} --count-only

## 2. Metadatos y tablas

Guarda la copia cruda de la API (`raw/api_pages/`) y arma `tables/observations.jsonl` y `tables/photos.jsonl` **sin** los registros que ya están en otras corridas. Muestra cuántos se saltan.

También asigna el **sector** a cada registro y muestra cuántos quedan en el sector, cuántos son de otro sector y cuántos quedan excluidos (zona excluida o fuera de sector).

In [ ]:
!python scripts/download.py --config {CONFIG} --steps metadata,tables

## 3. Fotos originales, versiones de 512 y 1024 px, y chequeo

In [ ]:
!python scripts/download.py --config {CONFIG} --steps photos,resize,check

## 4. Triaje (`triage_v2`)

Una consulta por registro con todas sus fotos (512 px): estado del registro (hueso, vivo, huevos…), número de huesos, si el conteo es posible, y qué hay en cada foto.

In [ ]:
!python scripts/triage.py --config {CONFIG}

## 5. Los 4 prompts de rasgos por foto

Una consulta por registro (1024 px): por foto, huesos visibles y, si hay exactamente 1, lado, erosión y calidad (v14: calidad descrita con `cover`/`focus` y decidida por una regla fija). 8 consultas por minuto. Cada versión escribe su propio archivo en `outputs/`.

In [ ]:
for v in VERSIONS:
    print(f"\n===== {v} =====")
    !python scripts/photo_traits.py --config {CONFIG} --prompt-version {v}

## 5b. Recuadro del hueso (`bonebox_v1`)

Una consulta por foto, **solo en fotos con 1 hueso según v14** (por eso va después del paso 5). El modelo solo da el recuadro; en la anotación (notebook 02) aparece ya dibujado y se acepta, se ajusta o se dibuja otro. Las fotos con varios huesos se encuadran a mano.

In [ ]:
!python scripts/bonebox.py --config {CONFIG}


## 6. Control: ¿terminó todo? (no muestra los resultados de los modelos)

In [ ]:
obs, pho = sepia.load_tables(cfg)
p = sepia.paths(cfg)
skipped = sepia.read_jsonl(p['logs'] / 'skipped_existing.jsonl')
print(f"Registros: {len(obs)} · fotos: {len(pho)} · saltados (ya en otra corrida): {len(skipped)}")

t = cfg.triage
tri = sepia.read_jsonl(sepia.triage.output_path(cfg, t['provider'], t['model'], t['prompt_version']))
if len(tri):
    ok = tri[tri['error'].isna()]['observation_id'].nunique()
    print(f"Triaje: {ok}/{len(obs)} registros sin error")

for v in VERSIONS:
    tc = {**cfg.photo_traits, 'prompt_version': v}
    raw = sepia.io.read_model_outputs(cfg, 'photo_traits', tc['provider'], tc['model'], v)
    if raw.empty:
        print(f"{v}: sin resultados todavia")
        continue
    last = raw.sort_values('saved_at').drop_duplicates('photo_id', keep='last')
    n_err = int(last['error'].notna().sum())
    print(f"{v}: {int(last['error'].isna().sum())}/{len(pho)} fotos sin error · con error: {n_err}"
          + (" (volver a correr el paso 5)" if n_err or len(last) < len(pho) else ""))
bb = sepia.load_bonebox(cfg, BONEBOX)
print(f"{BONEBOX}: {len(bb)} fotos con respuesta · con recuadro: "
      f"{int(bb['crop_box'].apply(lambda b: isinstance(b, dict)).sum()) if len(bb) else 0}")
print("Registros por sector:", obs['site'].value_counts(dropna=False).to_dict() if 'site' in obs else '-')
if 'excluded_reason' in obs:
    print("Excluidos:", obs['excluded_reason'].value_counts().to_dict())


## 7. Ver los resultados de los prompts, foto por foto

**Cuidado:** si después se van a anotar a ciegas fotos de esta corrida, no mirarlas aquí antes. *Show = all*: todas; *versions_differ*: solo donde los prompts no coinciden en el lado.

In [ ]:
from sepia.review import ReviewApp
ReviewApp(cfg, VERSIONS, only='all').show()